# Placing a MERFISH section in the Allen CCF

The same volume-to-section fit as [the STARmap notebook](starmap-allen3Datlas.ipynb), on a
denser section and a finer raster. It is the simpler of the two to write: this section needs
no anisotropic initialisation, so `initial_slice`, `initial_rotation` and `initial_scale` say
the whole starting guess and `initial_affine` never appears.

## Inputs

Cells as a points element, the atlas and its annotation volume as 3D images. The physical
placement lives on the elements, so nothing downstream builds coordinate axes by hand.

In [ ]:
import nrrd, numpy as np, pandas as pd, spatialdata as sd
from spatialdata.models import Image3DModel, PointsModel
from spatialdata.transformations import Scale, Sequence, Translation

cells = pd.read_csv('merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase'
                    '_Slice1_Replicate1_cell_metadata_S1R1.csv.gz')
xy = np.c_[cells['center_x'], cells['center_y']].astype(float)

atlas, hdr = nrrd.read('ara_nissl_50.nrrd')       # (z, y, x)
labels, _ = nrrd.read('annotation_50.nrrd')       # same frame, integer structure ids
voxel = tuple(np.diag(hdr['space directions']))

def centred(shape, spacing, axes):
    """The placement an atlas volume carries: voxel size, and the origin at its centre."""
    return Sequence([
        Scale(list(spacing), axes=axes),
        Translation(-(np.asarray(shape) - 1) * np.asarray(spacing) / 2, axes=axes),
    ])

sdata = sd.SpatialData(
    points={'cells': PointsModel.parse(xy)},
    images={'atlas': Image3DModel.parse(
        atlas[None].astype(float), dims=('c', 'z', 'y', 'x'),
        transformations={'global': centred(atlas.shape, voxel, ('z', 'y', 'x'))},
    )},
)
sdata

## The section

`rasterize_points` turns the centroids into a density image: each cell deposits unit mass
bilinearly, blurred once per scale, so total intensity is exactly the cell count. At `dx=10`
that mass spreads over 25x more pixels than the STARmap notebook's `dx=50` -- which is why the
intensity scale is worth looking at before choosing solver parameters.

In [ ]:
from squidpy.experimental.im import rasterize_points, sample_volume

rasterize_points(sdata, 'cells', dx=10.0, blur=1.0, key_added='section')
sdata['section']

## The fit

The section is close to coronal and close to atlas scale, so the initialisation is three
numbers: which slice to centre on, how far to rotate in plane, and one uniform scale.

Upstream also nudges the starting translation by a single landmark pair, worth about 42 um in
`x` and 7 um in `y` -- four pixels and under one. That is an initialisation, not an answer, and
the next cell checks the initialisation directly, so it is left out here rather than
reintroducing a hand-built affine to carry it.

In [ ]:
from squidpy.experimental.tl import align_stalign_volume

slice_index, rotation, scale = 177, 0.0, 0.9

# Upstream's own solver values, carried verbatim so this fit stays comparable with the pinned
# upstream run. They are in the *target's* intensity units -- and this raster never reaches 1,
# so `muA` sits outside the data entirely and the sigmas are wider than the whole dynamic
# range, which flattens the matching/artifact/background split into near-constants. They are
# in scale for a dx=50 raster, which spans 0 to ~13, not for the dx=10 used here.
#
# One channel, not three: upstream passes `muA=[3, 3, 3]` against a single-channel target and
# sums over the broadcast axis, making its effective widths sigma/sqrt(3) -- divergence D13.
SOLVER = dict(a=500.0, nt=4, sigmaM=2.0, sigmaA=2.0, sigmaB=2.0, muA=[3.0], muB=[0.0])

section = np.asarray(sdata['section']).squeeze()
print(f'section intensity spans {section.min():.3g} to {section.max():.3g}, '
      f"mean {section.mean():.3g} -- against muA={SOLVER['muA'][0]}, sigmaM={SOLVER['sigmaM']}")

### Is the initialisation in the right place?

`niter=0` returns the starting affine without fitting, so the initial guess can be looked at
through the same public route as the result. Worth doing: an initialisation that starts in the
wrong place produces a fit that never recovers, and the objective alone does not say so.

In [ ]:
guess = align_stalign_volume(
    sdata, image_key=('atlas', 'section'), niter=0,
    initial_slice=slice_index, initial_rotation=rotation, initial_scale=scale, **SOLVER,
)

def atlas_at(result):
    """The atlas resampled onto the section's plane, through a fit's own backward map."""
    plane = np.moveaxis(np.asarray(result.deformation_grid(direction='backward')), 0, -1)[0]
    sampled = sample_volume(atlas, result.ref_axes, plane.reshape(-1, 3)[:, ::-1])
    return sampled.reshape(plane.shape[:2])

# A fit reports the reference axes it read off the element, so the depth of the chosen slice
# needs no second derivation from the NRRD header -- and cannot disagree with the one used.
z_axis = np.asarray(guess.ref_axes[0])
initial_depth = np.asarray(guess.transform(xy))[:, 2]
print(f'initial guess places the section at z = {initial_depth.mean():.0f} um '
      f'(slice {slice_index} sits at {z_axis[slice_index]:.0f} um)')

In [ ]:
fit = align_stalign_volume(
    sdata, image_key=('atlas', 'section'), niter=2000,
    initial_slice=slice_index, initial_rotation=rotation, initial_scale=scale, **SOLVER,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

A fit that is still descending at its last iteration reports where it ran out of budget, not
where the optimum is -- so look at the trace before trusting the result.

In [ ]:
import matplotlib.pyplot as plt

energies = np.asarray(fit.energies)[: fit.n_iter]
tail = energies[-len(energies) // 10 :]
print(f'last tenth: mean {tail.mean():.0f}, spread {np.ptp(tail):.0f}')
plt.plot(energies, lw=0.8); plt.xlabel('iteration'); plt.ylabel('objective'); plt.grid(alpha=0.3)

## Where each cell lands

`transform` maps `(x, y)` section coordinates to `(x, y, z)` reference coordinates, evaluated at
each point rather than at the nearest raster cell. `sample_volume` then reads the annotation
volume there -- `order=0` because structure ids must not be interpolated.

In [ ]:
coords = np.asarray(fit.transform(xy))                 # (N, 3), (x, y, z) in microns
structure_id = sample_volume(labels, fit.ref_axes, coords, order=0).astype(int)

print(f'{len(coords)} cells placed, {np.unique(structure_id).size} distinct structures')
print(f'depth (z) spans {coords[:, 2].min():.0f} to {coords[:, 2].max():.0f} um')

Structure ids become acronyms through the Allen ontology.

In [ ]:
ontology = pd.read_csv('allen_ontology.csv').set_index('id')['acronym']
acronym = pd.Series(structure_id).map(ontology).fillna('unassigned')
acronym.value_counts().head(12)

## The aligned atlas over the section

In [ ]:
import matplotlib as mpl

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(section, cmap=mpl.cm.Blues)
ax[1].imshow(atlas_at(guess), cmap=mpl.cm.Reds)
ax[2].imshow(atlas_at(fit), cmap=mpl.cm.Reds)
ax[3].imshow(section, cmap=mpl.cm.Blues, alpha=0.9)
ax[3].imshow(atlas_at(fit), cmap=mpl.cm.Reds, alpha=0.3)
for a, t in zip(ax, ('MERFISH section', 'atlas at the initial guess',
                     'atlas after fitting', 'overlaid'), strict=True):
    a.set_title(t); a.set_xticks([]); a.set_yticks([])

## Cells coloured by structure

In [ ]:
keep = acronym.value_counts()
keep = keep[keep >= 50].index                       # a legend of singletons reads as noise
fig, ax = plt.subplots(figsize=(7, 6))
for region in keep:
    m = (acronym == region).to_numpy()
    ax.scatter(xy[m, 0], xy[m, 1], s=0.2, label=region)
ax.invert_yaxis(); ax.set_aspect('equal')
ax.legend(markerscale=20, fontsize=6, ncol=2, loc='center left', bbox_to_anchor=(1, 0.5))